# 03 — JSON feature store validation and the reference (RAG) index

**Objectives.**
1. Re-validate every record from notebook 02 against the schema, check strict JSON, and confirm that
   regeneration is deterministic. Export the JSON Schema.
2. Ingest the **authorized** reference documents declared in `config.yaml → research.rag.sources`,
   build a local retrieval index with chunk-level citations, and check that retrieval cannot leak case
   data or labels.

**Prerequisites.** Notebook 02 (`results/radiomics/manifest.json`).

**Outputs.** `results/radiomics/schema/lesion_record.schema.json`, `results/rag/index/`
(`chunks.jsonl`, `index_manifest.json`).

## Setup

In [1]:
import json
import os
import sys
from pathlib import Path

# Resolve the repo root whether the kernel starts in notebooks/ or at the root:
# config.yaml paths (dataset/, checkpoints/, results/) are relative to it.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "config.yaml").exists() and (p / "bus_research").is_dir())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.config import load_config
from utils.seed import get_device, set_seed

config = load_config(str(REPO_ROOT / "config.yaml"))
set_seed(config["seed"], config["deterministic"])
RESEARCH = config["research"]
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/mulatyazewchekol/Documents/My_Projects/Healthcare_Projects/Breast_Cancer_Detection


In [2]:
import bus_research

missing = bus_research.missing_dependencies()
print("Missing research dependencies:", missing or "none")
# If anything is missing, install the pinned optional set (see the comments in that file):
# %pip install -r requirements-research.txt

Missing research dependencies: none


## 1. Validate every stored record

In [3]:
from collections import Counter

import numpy as np
import pandas as pd

from bus_research.feature_store import (
    LESION_RECORD_SCHEMA, RecordValidationError, atomic_write_json, content_hash, dumps_strict, load_record)

rad_dir = REPO_ROOT / RESEARCH["radiomics"]["output_dir"]
manifest = json.loads((rad_dir / "manifest.json").read_text())
valid, invalid = 0, []
statuses = Counter()
for e in manifest["records"]:
    try:
        rec = load_record(rad_dir / e["record_path"])
        dumps_strict(rec)                       # raises on NaN/Infinity
        valid += 1
        statuses[(e["variant"], rec["radiomics"]["status"], rec["segmentation"]["quality_status"])] += 1
    except (RecordValidationError, ValueError) as exc:
        invalid.append((e["record_id"], str(exc)[:200]))
print(f"records in manifest: {manifest['n_records']} | valid: {valid} | invalid: {len(invalid)}")
print(invalid[:5] or "")
print(pd.Series(statuses).rename_axis(["variant", "radiomics", "qc"]).rename("n").to_string())
assert not invalid

records in manifest: 1951 | valid: 1951 | invalid: 0

variant         radiomics  qc  
gt_mask         ok         pass    653
unet_on_gt_box  ok         pass    620
                           warn     45
gt_mask         ok         warn     12
cascade         ok         pass    530
                           warn     91


In [4]:
schema_path = atomic_write_json(rad_dir / "schema" / "lesion_record.schema.json", LESION_RECORD_SCHEMA)
print("exported", schema_path.relative_to(REPO_ROOT))

example = load_record(rad_dir / manifest["records"][0]["record_path"])
shown = {k: v for k, v in example.items() if k != "radiomics"}
shown["radiomics"] = {**{k: v for k, v in example["radiomics"].items() if k not in ("features", "settings")},
                      "features": {fam: dict(list(vals.items())[:3]) | {"...": f"{len(vals)} total"}
                                   for fam, vals in example["radiomics"]["features"].items()}}
print(json.dumps(shown, indent=1)[:4000])

exported results/radiomics/schema/lesion_record.schema.json
{
 "schema_version": "1.0.0",
 "record_type": "breast_ultrasound_lesion",
 "record_id": "rec_f00e9d819f99",
 "image_id": "img_82858b989dd9",
 "lesion_id": "les_e6dfda1fa3dc",
 "image_metadata": {
  "width_px": 610,
  "height_px": 394,
  "channels": 3,
  "dtype": "uint8",
  "physical_pixel_spacing": null,
  "spacing_unit": "pixel",
  "image_modality": "ultrasound"
 },
 "segmentation": {
  "mask_area_px": 2110,
  "quality_status": "pass",
  "quality_issues": [],
  "model": "ground_truth_annotation",
  "checkpoint_version": null,
  "mask_source": "ground_truth",
  "activation": null,
  "threshold": null,
  "bbox_source": "ground_truth",
  "detection_confidence": null,
  "roi_bbox_xyxy": [
   518,
   167,
   596,
   214
  ]
 },
 "birads_descriptors": {
  "shape": {
   "value": null,
   "status": "not_assessed",
   "source": "not_annotated",
   "confidence": null,
   "note": ""
  },
  "orientation": {
   "value": null,
   "status":

**Value states in a record.** A finite number in `radiomics.features` is *observed*; an entry in
`radiomics.feature_status` is `invalid_nonfinite` or `unsupported`; `radiomics.status = "failed"` with
`errors` means extraction did not run. Each BI-RADS descriptor carries `status`
(`observed` / `not_assessed` / `unavailable` / `invalid`) and `source`. The schema rejects an
"observed" descriptor without a radiologist or model source, and any value on a non-observed one.

### Deterministic regeneration

In [5]:
from bus_research import birads
from bus_research.feature_store import build_record
from bus_research.radiomics_extract import RadiomicsSettings, extract_radiomics_features
from dataset.lesion_manifest import load_lesion_manifest
from preprocessing.mask_to_yolo import component_mask
from preprocessing.normalization import load_image_rgb

settings = RadiomicsSettings.from_config(config)
assert settings.settings_hash() == manifest["settings_hash"], "config changed since notebook 02 ran"
rows = load_lesion_manifest(Path(config["paths"]["splits_dir"]) / "val_lesions.json")[:3]
by_id = {e["record_id"]: e for e in manifest["records"]}
for row in rows:
    image = load_image_rgb(row["image_path"])
    mask = component_mask(row["source_mask_path"], row["component_label"]).astype(np.uint8)
    result = extract_radiomics_features(image, mask, {"roi_bbox": row["bbox"]}, settings)
    lesion_key = f"{Path(row['source_mask_path']).name}#{row['component_label']}"
    seg = {"model": "ground_truth_annotation", "checkpoint_version": None, "mask_source": "ground_truth",
           "activation": None, "threshold": None, "bbox_source": "ground_truth", "detection_confidence": None,
           "roi_bbox_xyxy": row["bbox"]}
    fresh = build_record(image_key=row["image_id"], lesion_key=lesion_key, image=image, radiomics_result=result,
                         segmentation=seg, descriptors=birads.empty_descriptors())
    stored = load_record(rad_dir / by_id[fresh["record_id"]]["record_path"])
    same = fresh["radiomics"]["features"] == stored["radiomics"]["features"]
    print(fresh["record_id"], "features identical to stored record:", same)
    assert same

rec_cad936f82ef4 features identical to stored record: True


rec_90cbfbbaea44

 features identical to stored record: True
rec_e8f5b3e73380 features identical to stored record: True


## 2. Reference ingestion

Only the sources declared in config are ingested, each with a title, citation, version and evidence
tier:

* **`birads-card`** (primary reference): the ACR BI-RADS Atlas 5th-edition *Quick Reference* card
  supplied with the project. **The full Atlas is not available**, so term definitions, management
  guidance and likelihood-of-malignancy tables from the Atlas are *missing evidence*.
* **`pyradiomics-docs`** (official documentation): feature definitions from the docstrings of the
  installed PyRadiomics version, which is the text its official documentation is generated from.
* **`project-*-notes`** (secondary project notes): the Stage 6 knowledge files. They are
  project-authored paraphrases, retrievable but labeled secondary, and never cited as primary evidence.

Ingestion refuses anything under `research.rag.forbidden_dirs` (`results/`, `dataset/splits/`,
`dataset/yolo_dataset/`), so no feature record, split or label can enter the index.

In [6]:
from bus_research import rag

rcfg = RESEARCH["rag"]
chunks, sources = rag.ingest_sources(rcfg["sources"], REPO_ROOT, rcfg["chunk_words"], rcfg["chunk_overlap_words"],
                                     forbidden_dirs=rcfg["forbidden_dirs"])
src_df = pd.DataFrame(sources)[["id", "evidence_tier", "kind", "version", "status", "n_chunks"]]
print(src_df.to_string(index=False))
print(f"\ntotal chunks: {len(chunks)}")
card = [c for c in chunks if c.source_id == "birads-card"]
print("\nexample primary chunk:", card[0].chunk_id, "| page", card[0].page)
print(card[0].text[:500])

                      id           evidence_tier              kind                        version   status  n_chunks
             birads-card       primary_reference               pdf 5th edition (card dated 07.15) ingested         7
        pyradiomics-docs  official_documentation python_docstrings        3.1.0a2.post11+g67d1f26 ingested       151
project-us-lexicon-notes secondary_project_notes          markdown                            NaN ingested        27
  project-category-notes secondary_project_notes          markdown                            NaN ingested         9

total chunks: 194

example primary chunk: birads-card:p1:0 | page 1
÷ ...... MAMMOGRAPHY Breast composition a. The breasts are almost entirely fatty b. There are scattered areas of fibroglandular density c. The breasts are heterogeneously dense, which may obscure small masses d. The breasts are extremely dense, which lowers the sensitivity of mammography Masses Shape Oval Round Irregular Margin Circumscribed Ob

## 3. Label-leakage guard

In [7]:
from dataset.splits import load_split

raw_ids = [r["image_id"] for s in ("train", "val", "test") for r in load_split(config["paths"]["splits_dir"], s)]
forbidden = (raw_ids + [e["record_id"] for e in manifest["records"]] + [e["image_id"] for e in manifest["records"]]
             + [e["lesion_id"] for e in manifest["records"]])
rag.assert_no_case_data(chunks, forbidden)
print(f"checked {len(chunks)} chunks against {len(forbidden)} case identifiers: no case data in the index")
for s in sources:
    if s.get("path"):
        assert not any((REPO_ROOT / s["path"]).resolve().is_relative_to((REPO_ROOT / d).resolve())
                       for d in rcfg["forbidden_dirs"])
print("no source lives under a forbidden directory:", rcfg["forbidden_dirs"])
try:
    rag.ingest_sources([{"id": "leak", "kind": "text", "path": "results/radiomics/manifest.json", "title": "x",
                         "citation": "x", "evidence_tier": "secondary_project_notes"}], REPO_ROOT,
                       forbidden_dirs=rcfg["forbidden_dirs"])
except PermissionError as exc:
    print("attempt to ingest the record manifest refused:", exc)

checked 194 chunks against 6500 case identifiers: no case data in the index
no source lives under a forbidden directory: ['results', 'dataset/splits', 'dataset/yolo_dataset']
attempt to ingest the record manifest refused: source leak is under a forbidden directory (case data/labels)


## 4. Build, save and reload the index

In [8]:
index = rag.build_index(chunks, sources, rcfg["chunk_words"], rcfg["chunk_overlap_words"],
                        embedding_model_id=rcfg.get("embedding_model_id"))
index.save(REPO_ROOT / rcfg["index_dir"])
reloaded = rag.ReferenceIndex.load(REPO_ROOT / rcfg["index_dir"])
print("index version:", reloaded.version, "| retrieval:", reloaded.manifest["retrieval"],
      "| chunks:", reloaded.manifest["n_chunks"])

index version: rag-c47fd3b06bdb | retrieval: bm25 | chunks: 194


## 5. Retrieval with citations

Every hit carries `chunk_id` (the citation key an LLM report must use), source title, citation,
version, page/section and evidence tier. The expected-source check below is a small sanity test
written by hand, not a retrieval benchmark.

In [9]:
queries = {
    "ultrasound mass margin spiculated angular microlobulated": "birads-card",
    "posterior features shadowing enhancement combined pattern": "birads-card",
    "assessment category probably benign suspicious highly suggestive of malignancy": "birads-card",
    "gray level co-occurrence matrix contrast": "pyradiomics-docs",
    "sphericity perimeter surface ratio 2D shape": "pyradiomics-docs",
    "elasticity assessment soft intermediate hard vascularity": "birads-card",
}
checks = []
for q, expected in queries.items():
    hits = reloaded.search(q, k=rcfg["top_k"])
    top_sources = [h["source_id"] for h in hits]
    checks.append({"query": q[:55], "expected": expected, "hit@k": expected in top_sources,
                   "top1": hits[0]["chunk_id"] if hits else None})
print(pd.DataFrame(checks).to_string(index=False))

print("\nprimary-tier evidence only:")
for h in reloaded.search("ultrasound margin not circumscribed indistinct", k=3, tiers=("primary_reference",)):
    print(f"  [{h['chunk_id']}] {h['citation']} (p{h['page']}, score {h['score']})")
assert reloaded.search("gray level co-occurrence matrix contrast", k=3) == \
       reloaded.search("gray level co-occurrence matrix contrast", k=3), "retrieval must be deterministic"

                                                  query         expected  hit@k                                                                top1
ultrasound mass margin spiculated angular microlobulate      birads-card   True project-us-lexicon-notes:us-suspicious-features-suspicious-ultras:0
posterior features shadowing enhancement combined patte      birads-card   True project-us-lexicon-notes:us-posterior-combined-posterior-features:0
assessment category probably benign suspicious highly s      birads-card   True                                                    birads-card:p2:2
               gray level co-occurrence matrix contrast pyradiomics-docs   True                                    pyradiomics-docs:radiomicsglcm:3
            sphericity perimeter surface ratio 2D shape pyradiomics-docs   True                      pyradiomics-docs:radiomicsshape2d-sphericity:0
elasticity assessment soft intermediate hard vascularit      birads-card   True                                 

## Limitations and missing evidence

* The full ACR BI-RADS Atlas (definitions, figures, management text, likelihood-of-malignancy
  tables) is **not available**. The card provides the vocabulary only. Likelihood ranges appear only in
  secondary project notes and must not be treated as authoritative.
* No BUSI dataset paper or annotation guideline is in the repository. Dataset documentation is
  therefore missing evidence. Add authorized copies to `research.rag.sources` to include them.
* No peer-reviewed breast-ultrasound radiomics literature is in the repository. None was scraped.
* Retrieval is BM25 (deterministic). Set `research.rag.embedding_model_id` to enable a BM25 +
  sentence-embedding hybrid. Retrieved text is evidence for reports, never a training label.